# XAI-MedCrossNet | VinDr-Mammo | Multi-View, Imbalance-Aware Pipeline

Techniques used to legitimately maximize AUC and handle class imbalance (no label leakage):
- **Multi-view fusion**: each sample is one breast (CC + MLO fused) — the biggest literature-supported AUC lever
- **Safe minority oversampling**: applied only inside each fold's training set, after the leakage-safe split — validation/test are never touched or duplicated
- **Focal Loss** with dynamic class weights (computed from real train-fold counts) — focuses learning on hard/minority examples
- Two reported thresholds: F1-optimal, and a sensitivity-targeted threshold (default: sensitivity ≥ 80%) — both tuned on out-of-fold predictions only, never on the test set
- MC-Dropout + horizontal-flip test-time augmentation at final inference
- Final metrics (AUC, balanced accuracy, sensitivity, precision, F1) computed strictly on the real held-out `split == 'test'` partition
- `NUM_WORKERS = 0` and `pathlib.Path` throughout for Windows/Linux/macOS reliability

**On accuracy and AUC targets:** this dataset is ~90% negative / ~10% positive at the breast level. A model that predicts "negative" for everyone already scores ~90% accuracy while catching zero malignant cases — so accuracy is not reported as a headline metric here. Realistic AUC on real held-out data for this task is **~0.78–0.87**; anything far above that on a genuine test set indicates a leakage bug, not model skill.

In [1]:
import os, sys, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd

import albumentations as A
from albumentations.pytorch import ToTensorV2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as tv_models

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score, f1_score, precision_score

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'[GPU] {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
    print('[MPS] Apple Silicon GPU')
else:
    device = torch.device('cpu')
    print('[CPU] No GPU detected')

# num_workers=0 keeps DataLoader multiprocessing safe across Windows/Linux/macOS
NUM_WORKERS = 0

print(f'Device: {device} | PyTorch: {torch.__version__}')


[GPU] NVIDIA GeForce RTX 5060 | VRAM: 8.5 GB
Device: cuda | PyTorch: 2.11.0+cu128


In [2]:
IMG_DIR = Path('./images_png')

breast_csv = Path('breast-level_annotations.csv')
if not breast_csv.exists():
    raise FileNotFoundError('breast-level_annotations.csv not found. Download from https://physionet.org/content/vindr-mammo/1.0.0/')

df_raw = pd.read_csv(breast_csv)
print(f'[OK] Loaded {breast_csv.name}: {len(df_raw):,} rows, {df_raw.study_id.nunique():,} studies')

MALIGNANT_BIRADS = {'BI-RADS 3', 'BI-RADS 4', 'BI-RADS 5', '3', '4', '5'}
df_raw['target_label'] = df_raw['breast_birads'].apply(
    lambda x: 1 if str(x).strip() in MALIGNANT_BIRADS else 0
).astype(int)

DENSITY_MAP = {'DENSITY A': 1, 'DENSITY B': 2, 'DENSITY C': 3, 'DENSITY D': 4}
df_raw['density_encoded'] = df_raw['breast_density'].map(DENSITY_MAP).fillna(0).astype(float) / 4.0
df_raw['view_position'] = df_raw['view_position'].astype(str).str.upper()
df_raw['patient_id'] = df_raw['study_id']

# Fuse the two views (CC + MLO) of each breast into a single 2-view sample,
# matching the multi-view protocol shown to improve AUC on VinDr-Mammo.
agg_spec = dict(
    breast_birads=('breast_birads', 'first'),
    target_label=('target_label', 'first'),
    density_encoded=('density_encoded', 'first'),
    patient_id=('patient_id', 'first'),
)
if 'split' in df_raw.columns:
    agg_spec['split'] = ('split', 'first')

meta = df_raw.groupby(['study_id', 'laterality']).agg(**agg_spec).reset_index()

pivot = (df_raw.pivot_table(index=['study_id', 'laterality'], columns='view_position',
                             values='image_id', aggfunc='first')
         .reset_index())
pivot = pivot.rename(columns={'CC': 'image_id_cc', 'MLO': 'image_id_mlo'})
for col in ['image_id_cc', 'image_id_mlo']:
    if col not in pivot.columns:
        pivot[col] = np.nan

df_breast = meta.merge(pivot, on=['study_id', 'laterality'], how='left')
df_breast['image_id_cc']  = df_breast['image_id_cc'].fillna(df_breast['image_id_mlo'])
df_breast['image_id_mlo'] = df_breast['image_id_mlo'].fillna(df_breast['image_id_cc'])
df_breast = df_breast.dropna(subset=['image_id_cc', 'image_id_mlo']).reset_index(drop=True)

print(f'[Multi-view] {len(df_breast):,} breast-level (CC+MLO) samples from {len(df_raw):,} images')

vc = df_breast['target_label'].value_counts().sort_index()
print('[Label Distribution]')
for lbl, cnt in vc.items():
    print(f'  Class {lbl}: {cnt:,} ({cnt/len(df_breast)*100:.1f}%)')

if 'split' in df_breast.columns:
    df_train_all = df_breast[df_breast['split'] == 'training'].reset_index(drop=True)
    df_test_held = df_breast[df_breast['split'] == 'test'].reset_index(drop=True)
    print(f'[Split] Training: {len(df_train_all):,} | Held-out test: {len(df_test_held):,}')
else:
    df_train_all = df_breast.reset_index(drop=True)
    df_test_held = pd.DataFrame()
    print(f'[Split] No split column found. Using all {len(df_train_all):,} rows for CV.')

print(f'[Dataset Ready] {len(df_train_all):,} training breasts | {df_train_all.patient_id.nunique():,} unique patients')


[OK] Loaded breast-level_annotations.csv: 20,000 rows, 5,000 studies
[Multi-view] 10,000 breast-level (CC+MLO) samples from 20,000 images
[Label Distribution]
  Class 0: 9,041 (90.4%)
  Class 1: 959 (9.6%)
[Split] Training: 8,000 | Held-out test: 2,000
[Dataset Ready] 8,000 training breasts | 4,000 unique patients


In [3]:
df = df_train_all.copy()

tab_cols = ['density_encoded']
TAB_DIM  = len(tab_cols)

print(f'[Tabular] Feature vector: {TAB_DIM} dims -> {tab_cols}')
print(f'[Tabular] NaN counts: {df[tab_cols].isna().sum().sum()}')


[Tabular] Feature vector: 1 dims -> ['density_encoded']
[Tabular] NaN counts: 0


In [4]:
from PIL import Image

class VinDrMultiViewDataset(Dataset):
    def __init__(self, df, img_dir, tab_matrix, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.tab_mat = np.nan_to_num(tab_matrix.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def _load(self, study_id, image_id):
        path = self.img_dir / f"{study_id}" / f"{image_id}.png"
        if not path.exists():
            path = self.img_dir / f"{image_id}.png"
        img = Image.open(path).convert('RGB')
        return np.array(img)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        study_id = row['study_id']

        cc_arr  = self._load(study_id, row['image_id_cc'])
        mlo_arr = self._load(study_id, row['image_id_mlo'])

        if self.transform:
            cc_arr  = self.transform(image=cc_arr)['image']
            mlo_arr = self.transform(image=mlo_arr)['image']

        return {
            'cc_img': cc_arr,
            'mlo_img': mlo_arr,
            'tabular': torch.tensor(self.tab_mat[idx], dtype=torch.float32),
            'label': torch.tensor(row['target_label'], dtype=torch.long)
        }

print('[OK] VinDrMultiViewDataset defined.')


[OK] VinDrMultiViewDataset defined.


In [5]:
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, embed_dim=256, num_heads=4, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, img_feat, tab_feat):
        Q = img_feat.unsqueeze(1)
        K = tab_feat.unsqueeze(1)
        V = tab_feat.unsqueeze(1)
        attn_out, _ = self.attn(Q, K, V)
        attn_out = attn_out.squeeze(1)
        return self.norm(img_feat + self.dropout(attn_out))


class TabularMLP(nn.Module):
    def __init__(self, in_dim, embed_dim=256, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, embed_dim),
            nn.LayerNorm(embed_dim),
        )

    def forward(self, x):
        return self.net(x)


class XAIMedCrossNetMultiView(nn.Module):
    """Twin-view (CC + MLO) ConvNeXt-Small backbone with tabular cross-attention fusion."""
    def __init__(self, tab_dim, embed_dim=256, num_classes=2, dropout_p=0.3):
        super().__init__()

        backbone = tv_models.convnext_small(weights=tv_models.ConvNeXt_Small_Weights.DEFAULT)
        in_feats = backbone.classifier[2].in_features
        backbone.classifier[2] = nn.Identity()
        self.backbone = backbone

        self.img_proj = nn.Sequential(
            nn.Linear(in_feats * 2, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(dropout_p),
            nn.Linear(512, embed_dim),
            nn.LayerNorm(embed_dim),
        )

        self.tab_proj = TabularMLP(tab_dim, embed_dim, dropout_p)
        self.cross_attn = MultiHeadCrossAttention(embed_dim, num_heads=4, dropout=0.1)
        self.mc_dropout = nn.Dropout(dropout_p)
        self.classifier = nn.Linear(embed_dim, num_classes)

        self._init_new_layers()

    def _init_new_layers(self):
        for m in [self.img_proj, self.tab_proj, self.cross_attn, self.classifier]:
            for sub in (m.modules() if hasattr(m, 'modules') else [m]):
                if isinstance(sub, nn.Linear):
                    nn.init.xavier_uniform_(sub.weight)
                    if sub.bias is not None:
                        nn.init.zeros_(sub.bias)

    def forward(self, cc_img, mlo_img, tabular):
        feat_cc  = self.backbone(cc_img)
        feat_mlo = self.backbone(mlo_img)

        feat_visual = torch.cat([feat_cc, feat_mlo], dim=1)
        img_emb = self.img_proj(feat_visual)

        tab_emb = self.tab_proj(tabular)
        fused = self.cross_attn(img_emb, tab_emb)
        return self.classifier(self.mc_dropout(fused))


print('[OK] Multi-view Hybrid Architecture defined with ConvNeXt-Small.')


[OK] Multi-view Hybrid Architecture defined with ConvNeXt-Small.


In [6]:
def enable_only_dropout(model: nn.Module) -> None:
    model.eval()
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device) -> tuple:
    model.eval()
    all_preds, all_labels = [], []
    for batch in loader:
        cc_imgs  = batch['cc_img'].to(device, non_blocking=True)
        mlo_imgs = batch['mlo_img'].to(device, non_blocking=True)
        tabs     = batch['tabular'].to(device, non_blocking=True)
        labels   = batch['label'].cpu().numpy()

        probs = F.softmax(model(cc_imgs, mlo_imgs, tabs), dim=1)[:, 1].cpu().numpy()
        all_preds.extend(probs)
        all_labels.extend(labels)

    preds  = np.array(all_preds)
    labels = np.array(all_labels)
    binary = (preds >= 0.5).astype(int)

    acc  = accuracy_score(labels, binary)
    auc  = roc_auc_score(labels, preds) if len(np.unique(labels)) > 1 else 0.5
    sens = recall_score(labels, binary, zero_division=0)
    return acc, auc, sens, preds, labels


@torch.no_grad()
def evaluate_mc_dropout_tta(model: nn.Module, loader: DataLoader, device: torch.device,
                             n_samples: int = 10, tta_flip: bool = True) -> tuple:
    """MC-Dropout uncertainty estimation + horizontal-flip test-time augmentation."""
    enable_only_dropout(model)
    all_preds, all_labels, all_vars = [], [], []

    for batch in loader:
        cc_imgs  = batch['cc_img'].to(device, non_blocking=True)
        mlo_imgs = batch['mlo_img'].to(device, non_blocking=True)
        tabs     = batch['tabular'].to(device, non_blocking=True)
        labels   = batch['label'].cpu().numpy()

        views = [(cc_imgs, mlo_imgs)]
        if tta_flip:
            views.append((torch.flip(cc_imgs, dims=[3]), torch.flip(mlo_imgs, dims=[3])))

        mc_probs = []
        for cc_v, mlo_v in views:
            mc_probs.extend([
                F.softmax(model(cc_v, mlo_v, tabs), dim=1)[:, 1].cpu().numpy()
                for _ in range(n_samples)
            ])
        mc_probs = np.stack(mc_probs)

        mean_prob = mc_probs.mean(axis=0)
        var_prob  = mc_probs.var(axis=0)

        all_preds.extend(mean_prob)
        all_labels.extend(labels)
        all_vars.extend(var_prob)

    return np.array(all_preds), np.array(all_labels), np.array(all_vars)


def build_optimizer(model: nn.Module) -> torch.optim.Optimizer:
    return torch.optim.AdamW([
        {'params': model.backbone.parameters(),   'lr': 1e-5, 'name': 'backbone'},
        {'params': model.img_proj.parameters(),   'lr': 2e-4, 'name': 'img_proj'},
        {'params': model.tab_proj.parameters(),   'lr': 2e-4, 'name': 'tab_proj'},
        {'params': model.cross_attn.parameters(), 'lr': 2e-4, 'name': 'cross_attn'},
        {'params': model.classifier.parameters(), 'lr': 2e-4, 'name': 'classifier'},
    ], weight_decay=1e-2)


class FocalLoss(nn.Module):
    """Cross-entropy variant that down-weights easy (well-classified) examples,
    which helps minority-class recall more than class-weighting alone."""
    def __init__(self, alpha: torch.Tensor, gamma: float = 2.0, label_smoothing: float = 0.05):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.alpha,
                              label_smoothing=self.label_smoothing, reduction='none')
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()


def build_criterion(train_labels: np.ndarray, device: torch.device, gamma: float = 2.0):
    class_counts = np.bincount(train_labels)
    class_weights = len(train_labels) / (len(class_counts) * class_counts)
    weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
    return FocalLoss(alpha=weights, gamma=gamma, label_smoothing=0.05)


print('[OK] Training helpers defined.')


[OK] Training helpers defined.


In [7]:
train_transform = A.Compose([
    A.Resize(512, 512),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.Affine(rotate=(-8, 8), translate_percent=(0.0, 0.05), scale=(0.95, 1.05), p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(512, 512),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

print('[OK] Multi-view transforms defined.')


[OK] Multi-view transforms defined.


In [ ]:
import gc

EPOCHS      = 20
BATCH_SIZE  = 8
ACCUM_STEPS = 4
N_FOLDS     = 5
EMBED_DIM   = 256
DROPOUT_P   = 0.3

sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_best = []
oof_preds, oof_labels = [], []

for fold, (tr_idx, vl_idx) in enumerate(sgkf.split(df, df['target_label'], df['patient_id'])):
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()

    print(f'\n{"="*65}')
    print(f' FOLD {fold+1}/{N_FOLDS} | {len(tr_idx):,} train / {len(vl_idx):,} val')
    print(f'{"="*65}')

    tr_df = df.iloc[tr_idx].copy()
    vl_df = df.iloc[vl_idx].copy()

    train_pats = set(tr_df['patient_id'])
    val_pats = set(vl_df['patient_id'])
    assert len(train_pats & val_pats) == 0, '[FATAL] Patient leakage detected!'

    pos_df = tr_df[tr_df['target_label'] == 1]
    neg_df = tr_df[tr_df['target_label'] == 0]
    target_pos_ratio = 0.30
    n_pos_needed = int(len(neg_df) * target_pos_ratio / (1 - target_pos_ratio))
    if 0 < len(pos_df) < n_pos_needed:
        reps = n_pos_needed // len(pos_df)
        remainder = n_pos_needed % len(pos_df)
        extra = pos_df.sample(remainder, random_state=SEED, replace=(remainder > len(pos_df)))
        pos_df = pd.concat([pos_df] * reps + [extra], ignore_index=True)
    tr_df = pd.concat([neg_df, pos_df], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
    print(f' [Balance] Train set oversampled to {tr_df["target_label"].mean()*100:.1f}% positive '
          f'({len(tr_df):,} rows, val set untouched)')

    scaler = StandardScaler()
    tr_df[tab_cols] = scaler.fit_transform(tr_df[tab_cols])
    vl_df[tab_cols] = scaler.transform(vl_df[tab_cols])

    tr_ds = VinDrMultiViewDataset(tr_df, IMG_DIR, tr_df[tab_cols].values, train_transform)
    vl_ds = VinDrMultiViewDataset(vl_df, IMG_DIR, vl_df[tab_cols].values, val_transform)

    tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=NUM_WORKERS, pin_memory=(device.type == 'cuda'), drop_last=True)
    vl_loader = DataLoader(vl_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=(device.type == 'cuda'))

    model = XAIMedCrossNetMultiView(tab_dim=TAB_DIM, embed_dim=EMBED_DIM, dropout_p=DROPOUT_P).to(device)
    optimizer = build_optimizer(model)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-7)
    criterion = build_criterion(tr_df['target_label'].values, device)
    scaler_amp = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

    best_auc, best_epoch = 0.0, 0
    best_val_preds, best_val_labels = None, None
    patience, no_improve = 5, 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        optimizer.zero_grad(set_to_none=True)

        for step, batch in enumerate(tr_loader):
            cc_imgs  = batch['cc_img'].to(device, non_blocking=True)
            mlo_imgs = batch['mlo_img'].to(device, non_blocking=True)
            tabs     = batch['tabular'].to(device, non_blocking=True)
            labels   = batch['label'].to(device, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                logits = model(cc_imgs, mlo_imgs, tabs)
                loss   = criterion(logits, labels) / ACCUM_STEPS

            scaler_amp.scale(loss).backward()

            if (step + 1) % ACCUM_STEPS == 0:
                scaler_amp.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler_amp.step(optimizer)
                scaler_amp.update()
                optimizer.zero_grad(set_to_none=True)

            running_loss += loss.item() * ACCUM_STEPS * labels.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)

        scheduler.step()
        epoch_loss = running_loss / total
        epoch_acc  = correct / total

        vl_acc, vl_auc, sens, vl_preds, vl_labels = evaluate(model, vl_loader, device)

        note = ''
        if vl_auc > best_auc:
            best_auc, best_epoch = vl_auc, epoch
            best_val_preds, best_val_labels = vl_preds, vl_labels
            no_improve = 0
            torch.save({
                'epoch': epoch, 'fold': fold + 1, 'state_dict': model.state_dict(),
                'val_auc': vl_auc, 'val_acc': vl_acc, 'scaler': scaler, 'tab_cols': tab_cols,
            }, f'best_vindr_mv_f{fold+1}.pth')
            note = 'SAVED'
        else:
            no_improve += 1

        print(f' Ep {epoch:3d} | loss {epoch_loss:7.4f} | tr_acc {epoch_acc*100:6.2f}% | '
              f'vl_acc {vl_acc*100:6.2f}% | AUC {vl_auc:6.4f} | sens {sens:6.4f} | {note}')

        if no_improve >= patience:
            print(f' [Early stop] No AUC improvement for {patience} epochs.')
            break

    print(f'\n Fold {fold+1} | Best Val AUC: {best_auc:.4f} (epoch {best_epoch})')
    fold_best.append({'fold': fold+1, 'auc': best_auc, 'epoch': best_epoch})
    oof_preds.extend(best_val_preds)
    oof_labels.extend(best_val_labels)

fold_aucs = [r['auc'] for r in fold_best]
print(f'\n{"="*65}\n Cross-Validation Summary')
for r in fold_best:
    print(f' Fold {r["fold"]}: AUC = {r["auc"]:.4f} (best epoch {r["epoch"]})')
print(f' Mean AUC: {np.mean(fold_aucs):.4f} +/- {np.std(fold_aucs):.4f}')
print(f' OOF AUC:  {roc_auc_score(oof_labels, oof_preds):.4f}')
print(f'{"="*65}')



 FOLD 1/5 | 6,400 train / 1,600 val
 [Balance] Train set oversampled to 30.0% positive (8,268 rows, val set untouched)


In [ ]:
N_MC_SAMPLES = 10
MIN_SENSITIVITY_TARGET = 0.80

oof_arr = np.array(oof_preds)

best_thresh, best_f1 = 0.5, 0.0
for thresh in np.arange(0.10, 0.90, 0.02):
    y_pred = (oof_arr >= thresh).astype(int)
    f1 = f1_score(oof_labels, y_pred, zero_division=0)
    if f1 > best_f1:
        best_f1, best_thresh = f1, thresh

sens_thresh, sens_prec = 0.5, 0.0
for thresh in np.arange(0.90, 0.05, -0.02):
    y_pred = (oof_arr >= thresh).astype(int)
    sens = recall_score(oof_labels, y_pred, zero_division=0)
    if sens >= MIN_SENSITIVITY_TARGET:
        sens_thresh = thresh
        sens_prec = precision_score(oof_labels, y_pred, zero_division=0)
        break

print(f'[Threshold: F1-optimal]         {best_thresh:.2f} (OOF F1 {best_f1:.4f})')
print(f'[Threshold: sensitivity>={MIN_SENSITIVITY_TARGET:.0%}] {sens_thresh:.2f} (OOF precision at this threshold: {sens_prec:.4f})')

if len(df_test_held) == 0:
    raise RuntimeError('No held-out test split available in this dataset to report final metrics on.')

ensemble_probs, all_labels = [], None

for fold in range(1, N_FOLDS + 1):
    ckpt_path = f'best_vindr_mv_f{fold}.pth'
    if not os.path.exists(ckpt_path):
        continue

    checkpoint = torch.load(ckpt_path, map_location=device)
    fold_scaler = checkpoint['scaler']

    test_df = df_test_held.copy()
    test_df[tab_cols] = fold_scaler.transform(test_df[tab_cols])
    test_ds = VinDrMultiViewDataset(test_df, IMG_DIR, test_df[tab_cols].values, val_transform)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=(device.type == 'cuda'))

    model = XAIMedCrossNetMultiView(tab_dim=TAB_DIM, embed_dim=EMBED_DIM, dropout_p=DROPOUT_P).to(device)
    model.load_state_dict(checkpoint['state_dict'])

    probs, labels, _ = evaluate_mc_dropout_tta(model, test_loader, device, n_samples=N_MC_SAMPLES, tta_flip=True)
    ensemble_probs.append(probs)
    all_labels = labels
    print(f' [OK] Fold {fold} inference done (val AUC was {checkpoint.get("val_auc", 0.0):.4f})')

y_prob = np.mean(ensemble_probs, axis=0)
y_true = all_labels
ens_auc = roc_auc_score(y_true, y_prob)

print(f'\n{"="*65}')
print(f' FINAL HELD-OUT TEST PERFORMANCE ({N_MC_SAMPLES} MC samples + flip TTA, {len(ensemble_probs)}-fold ensemble)')
print(f'{"-"*65}')
print(f' ROC-AUC (threshold-independent) : {ens_auc:.4f}')
print(f'{"-"*65}')

for label, thresh in [('F1-optimal', best_thresh), (f'Sensitivity>={MIN_SENSITIVITY_TARGET:.0%} target', sens_thresh)]:
    y_pred = (y_prob >= thresh).astype(int)
    acc  = accuracy_score(y_true, y_pred)
    sens = recall_score(y_true, y_pred, zero_division=0)
    prec = precision_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    bal_acc = (sens + recall_score(1 - y_true, 1 - y_pred, zero_division=0)) / 2
    print(f' [{label}] threshold={thresh:.2f}')
    print(f'   Accuracy {acc*100:.2f}% | Balanced Acc {bal_acc*100:.2f}% | '
          f'Sensitivity {sens*100:.2f}% | Precision {prec*100:.2f}% | F1 {f1:.4f}')

print(f'{"="*65}')
